# Risk-Sensitive Q-Learning for Option Hedging

This notebook reproduces the main result: a tabular Q-learning agent that beats Black-Scholes delta hedging on the risk-adjusted objective under transaction costs.

**Configuration:**
- CCHP dual-Q formulation (Q1 tracks `E[cost]`, Q2 tracks `E[cost²]`)
- Risk objective: `Y = E[cost] + c * sqrt(Var[cost])` with `c = 1.5`
- State: `(time_bin, moneyness_bin, gap_bin = pos - BS_delta)`
- Action space: `{-0.1, -0.02, 0.0, 0.02, 0.1}`
- Narrow gap grid: `[-0.2, 0.2]` with 31 buckets
- Two-phase training: 30% exploring starts, then 70% on-distribution
- Post-hoc no-trade bias `nt_bonus = 0.0005` to discourage spurious trading
- Scale: 800 batches × 64 parallel episodes = 51,200 total episodes (≈90 seconds on a laptop)

## 1. Imports and configuration

In [ ]:
from exp_framework import (cchp_dual_q_train, make_cchp_policy, make_bs_policy,
                           simulate_paths_batch, bs_delta_vec, bs_price_vec,
                           evaluate_policy)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

# Market & option parameters
K, T, S0 = 100.0, 1.0, 100.0      # strike, maturity (years), spot
sigma, r = 0.2, 0.0                # volatility, risk-free rate
dt, kappa = 1/252, 0.01            # step size (daily), proportional transaction cost
n_steps = int(round(T/dt))

# RL hyperparameters
c_risk = 1.5                       # risk aversion in Y = mean + c*std
nt_bonus = 0.0005                  # no-trade bias at evaluation (sweet spot)
actions = np.array([-0.1, -0.02, 0.0, 0.02, 0.1])

# State discretization
n_time, n_money, n_gap = 52, 40, 31
m_min, m_max = 0.5, 1.5
g_min, g_max = -0.2, 0.2
m_edges = np.linspace(m_min, m_max, n_money + 1)
g_edges = np.linspace(g_min, g_max, n_gap + 1)

# Training schedule
n_batches = 800
batch_size = 64
total_episodes = n_batches * batch_size
alpha_start, alpha_end = 0.05, 0.002
eps_start, eps_end = 1.0, 0.02
exploring_starts_frac = 0.3

# Derived values
init_pos = float(bs_delta_vec(np.array([S0]), K, np.array([T]), r, sigma)[0])
option_price = float(bs_price_vec(np.array([S0]), K, np.array([T]), r, sigma)[0])

print(f"Option:   K={K}, T={T}, sigma={sigma}, r={r}")
print(f"Market:   S0={S0}, kappa={kappa}, dt=1/{int(1/dt)}")
print(f"BS price: ${option_price:.4f}")
print(f"Init pos: BS delta at t=0 = {init_pos:.4f}")
print(f"Q-table:  {n_time}x{n_money}x{n_gap}x{len(actions)} = "
      f"{n_time*n_money*n_gap*len(actions):,} entries")
print(f"Training: {n_batches} batches x {batch_size} parallel = {total_episodes:,} episodes")

## 2. Train the CCHP dual-Q agent

Takes ~90 seconds on a typical laptop. During training you'll see the per-batch average episode cost (lower = better) and state-space coverage (what fraction of Q-table cells have been visited at least once).

In [ ]:
t0 = time.time()
Q1, Q2, visits, train_costs = cchp_dual_q_train(
    K, T, S0, sigma, r, dt, kappa, init_pos,
    actions, c_risk,
    n_time, n_money, n_gap, m_edges, g_edges,
    n_batches, batch_size,
    alpha_start, alpha_end, eps_start, eps_end,
    exploring_starts_frac=exploring_starts_frac,
    seed=42, print_every=100,
)
train_time = time.time() - t0
print(f"\nTraining complete in {train_time:.0f}s ({train_time/60:.1f}min)")
print(f"Q1 coverage: {100*np.count_nonzero(Q1)/Q1.size:.1f}%")
print(f"Avg visits per populated cell: {visits.sum()/np.count_nonzero(visits):.1f}")

### Training curve

The spike around batch 240 marks the transition from exploring-starts (randomized initial state) to on-distribution training (always starting at the real initial state). This is expected behaviour.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(train_costs, alpha=0.3, color='steelblue', label='Per-batch avg episode cost')
rm = pd.Series(train_costs).rolling(50).mean()
ax.plot(rm, color='darkblue', lw=2, label='Rolling mean (50 batches)')
ax.axvline(int(exploring_starts_frac * n_batches), color='crimson', ls='--', alpha=0.6,
          label='End of exploring starts')
ax.set_xlabel('Batch')
ax.set_ylabel('Mean episode cost (normalized)')
ax.set_title(f'Training Progress ({total_episodes:,} total episodes)')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('final_training.png', dpi=130)
plt.show()

## 3. Build policies and set up evaluation

We compare six strategies:
- **No hedge**: never trade (lower bound on what a hedge must do better than)
- **BS daily / weekly / biweekly**: rebalance Black-Scholes delta every 1 / 5 / 10 days
- **RL (nt=0)**: trained Q-tables with no-trade bias turned off
- **RL + no-trade bias**: trained Q-tables with `nt_bonus=0.0005`

In [ ]:
n_eval = 10_000
rng = np.random.default_rng(777)
paths = simulate_paths_batch(S0, sigma, r, dt, n_steps, n_eval, rng)
times = np.arange(n_steps + 1) * dt

pol_rl = make_cchp_policy(Q1, Q2, actions, c_risk, K, r, sigma, T,
                         n_time, m_edges, g_edges, n_money, n_gap,
                         nt_bonus=nt_bonus)
pol_rl_no_bias = make_cchp_policy(Q1, Q2, actions, c_risk, K, r, sigma, T,
                                  n_time, m_edges, g_edges, n_money, n_gap,
                                  nt_bonus=0.0)
pol_bs = make_bs_policy(K, r, sigma)
pol_no = lambda S, tau, pos: pos  # never trade


def full_evaluate(policy, rebal_every=1):
    """Run policy over all eval paths, return (hedge_cost_per_path, num_trades_per_path)."""
    n_trials = paths.shape[1]
    costs = np.zeros(n_trials)
    n_trades = np.zeros(n_trials, dtype=int)
    pos_prev = np.full(n_trials, init_pos)
    ttm0 = np.full(n_trials, T)
    pos_next = policy(paths[0], ttm0, pos_prev)
    n_trades += (np.abs(pos_next - pos_prev) > 1e-8).astype(int)
    for t in range(1, n_steps + 1):
        S_now, S_prev = paths[t], paths[t-1]
        tau_now = max(0.0, T - times[t])
        tau_prev = max(0.0, T - times[t-1])
        C_now = bs_price_vec(S_now, K, tau_now, r, sigma)
        C_prev = bs_price_vec(S_prev, K, tau_prev, r, sigma)
        step = (S_now - S_prev)*pos_prev - np.abs(pos_next - pos_prev)*S_now*kappa - C_now + C_prev
        if t == n_steps:
            step -= pos_next * S_now * kappa
        costs += step
        if t < n_steps:
            pos_prev = pos_next
            if t % rebal_every == 0:
                ttm = np.full(n_trials, max(0.0, T - times[t]))
                pos_next = policy(paths[t], ttm, pos_prev)
                n_trades += (np.abs(pos_next - pos_prev) > 1e-8).astype(int)
    return -costs, n_trades  # hedge cost (>0 = money lost)

## 4. Evaluate all strategies

In [ ]:
strategies = {
    "No hedge":            (pol_no, 1),
    "BS daily":            (pol_bs, 1),
    "BS weekly":           (pol_bs, 5),
    "BS biweekly":         (pol_bs, 10),
    "RL (nt=0)":           (pol_rl_no_bias, 1),
    "RL + no-trade bias":  (pol_rl, 1),
}

rows = []
costs_by_name = {}
for name, (pol, re) in strategies.items():
    h, nt = full_evaluate(pol, rebal_every=re)
    costs_by_name[name] = h
    mean, std = h.mean(), h.std()
    cvar = h[h >= np.quantile(h, 0.95)].mean()
    Y = mean + c_risk*std
    rows.append({
        'Strategy': name,
        'Mean $': mean, 'Std $': std, 'CVaR-95% $': cvar,
        f'Y = Mean + {c_risk}*Std': Y,
        'Mean %': 100*mean/option_price,
        'Trades/ep': nt.mean(),
    })
df = pd.DataFrame(rows).set_index('Strategy')

print(f"Final Results  (10,000 MC eval paths, BS option price = ${option_price:.4f})")
print("=" * 90)
df.style.format("{:.3f}")

## 5. Cost distribution histogram

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = {'BS daily': 'steelblue', 'BS weekly': 'gray', 'BS biweekly': 'lightgray',
          'RL + no-trade bias': 'crimson', 'RL (nt=0)': 'orange', 'No hedge': 'lightblue'}
for name in ['BS daily', 'BS weekly', 'RL (nt=0)', 'RL + no-trade bias']:
    ax.hist(costs_by_name[name], bins=80, alpha=0.5,
           label=name, color=colors.get(name, 'black'), density=True)
ax.axvline(0, color='k', lw=0.5, ls='--')
ax.set_xlabel('Hedge Cost ($)')
ax.set_ylabel('Density')
ax.set_title(f'Hedge Cost Distributions (BS call = ${option_price:.2f})')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('final_histogram.png', dpi=130)
plt.show()

## 6. Cost-risk frontier

The lower-left corner is best: low expected cost AND low risk. BS weekly/biweekly sit on the efficient frontier of the Black-Scholes strategies; the RL + no-trade-bias policy lies *below* the frontier — lower mean cost at comparable risk.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for name, h in costs_by_name.items():
    c = 'crimson' if 'RL +' in name else ('orange' if 'RL (' in name
                                         else ('steelblue' if 'BS daily' in name
                                              else ('gray' if 'BS weekly' in name
                                                   else ('lightgray' if 'biweekly' in name
                                                        else 'lightblue'))))
    sz = 200 if 'RL +' in name else 120
    ax.scatter(h.std(), h.mean(), s=sz, color=c, zorder=5)
    ax.annotate(name, (h.std(), h.mean()), xytext=(8, 4),
               textcoords='offset points', fontsize=10)
ax.set_xlabel('Hedge Cost Std ($)  ---  RISK')
ax.set_ylabel('Mean Hedge Cost ($)  ---  EXPECTED COST')
ax.set_title('Cost-Risk Frontier (lower-left = better)')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('final_frontier.png', dpi=130)
plt.show()

## 7. Trades per episode

How often does each strategy actually rebalance? BS daily trades 245 times/year; the RL agent discovers that ~10 trades/year is enough.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for name, (pol, re) in [("BS daily", (pol_bs, 1)), ("BS weekly", (pol_bs, 5)),
                        ("RL (nt=0)", (pol_rl_no_bias, 1)),
                        ("RL + no-trade bias", (pol_rl, 1))]:
    _, trades = full_evaluate(pol, rebal_every=re)
    ax.hist(trades, bins=40, alpha=0.5, label=name, density=True)
ax.set_xlabel('Number of trades per episode')
ax.set_ylabel('Density')
ax.set_title('How often does each policy rebalance?')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('final_trades.png', dpi=130)
plt.show()

## 8. Learned no-trade band structure

The left panel shows the trade size chosen as a function of the gap (position minus BS delta) at different times-to-maturity. Inside the band |gap| < ~0.18 the agent does nothing; outside it snaps the position back toward BS delta. This is the classical Davis-Panas-Zariphopoulou (1993) no-transaction band, rediscovered from data.

The right panel shows a sample hedging trajectory: the blue BS-daily policy rebalances constantly, while the red RL policy makes a handful of discrete, well-timed trades.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: trades chosen as function of gap, at different TTMs
gap_range = np.linspace(-0.3, 0.3, 400)
S_grid = np.full(gap_range.shape, S0)

for ttm_label, ttm in [('TTM=9mo', 0.75), ('TTM=6mo', 0.5), ('TTM=3mo', 0.25), ('TTM=1mo', 1/12)]:
    tau_arr = np.full_like(gap_range, ttm)
    bsd = bs_delta_vec(S_grid, K, tau_arr, r, sigma)
    pos_arr = bsd + gap_range
    valid = (pos_arr >= 0) & (pos_arr <= 1)
    trades = pol_rl(S_grid[valid], tau_arr[valid], pos_arr[valid])
    delta_trades = trades - pos_arr[valid]
    axes[0].plot(gap_range[valid], delta_trades, lw=1.6, label=ttm_label)

axes[0].axhline(0, color='k', lw=0.5, ls='--')
axes[0].axvline(0, color='crimson', lw=0.5, ls='--', label='gap=0 (at BS delta)')
axes[0].set_xlabel('Gap = position - BS delta')
axes[0].set_ylabel('Trade size (change in position)')
axes[0].set_title('Learned no-trade band structure (ATM)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Right: sample path trajectory
t_plot = np.arange(n_steps + 1) * dt
path_idx = 100
S_sample = paths[:, path_idx]

bs_positions = [init_pos]
for t in range(1, n_steps + 1):
    bsd_t = bs_delta_vec(np.array([S_sample[t]]), K, np.array([T - t*dt]), r, sigma)[0]
    bs_positions.append(bsd_t)

rl_positions = [init_pos]
pos_prev_scalar = init_pos
for t in range(1, n_steps + 1):
    tau = max(0.0, T - (t-1)*dt)
    new_pos = pol_rl(np.array([S_sample[t-1]]), np.array([tau]),
                    np.array([pos_prev_scalar]))[0]
    rl_positions.append(new_pos)
    pos_prev_scalar = new_pos

axes[1].plot(t_plot, S_sample/K, 'k-', alpha=0.5, label='S/K')
axes[1].plot(t_plot, bs_positions, 'steelblue', lw=2, label='BS delta (daily)')
axes[1].plot(t_plot, rl_positions, 'crimson', lw=2, label='RL position (nt bias)')
axes[1].set_xlabel('Time (years)')
axes[1].set_ylabel('Position / Moneyness')
axes[1].set_title(f'Example hedge trajectory (path #{path_idx})')
axes[1].legend(); axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig('final_policy.png', dpi=130)
plt.show()

## 9. Save trained Q-tables

Save the Q-tables to disk so you can reload and re-evaluate without retraining.

In [ ]:
np.savez_compressed('final_Q_tables.npz',
                    Q1=Q1, Q2=Q2, visits=visits,
                    actions=actions, m_edges=m_edges, g_edges=g_edges)
print("Q-tables saved to final_Q_tables.npz")
print("\nPlots saved: final_training.png, final_histogram.png, final_frontier.png,")
print("             final_trades.png, final_policy.png")